# 07 · Decision trees & random forests

A **decision tree** greedily picks, at each node, the split (a numeric threshold or a category) that most reduces impurity in the two resulting groups. It recurses until a stopping rule (`max_depth`, `min_samples_to_split`, or a pure node), then each leaf predicts from the training rows that landed there. `impurity_measure` selects the task:

- **Classification** — `"gini"` or `"entropy"`; leaf predicts the **majority class**.
- **Regression** — `"mse"` (variance reduction); leaf predicts the **mean** target.

A single tree is **greedy and high-variance**: small data changes can flip the best split near the root and cascade into a different tree. A **random forest** reduces variance by growing many trees on **bootstrapped** row samples, each restricted to a **random subset of columns** (`n_feat`), then combining predictions (majority vote / averaging). Decorrelated trees average out noise that a single tree can't.

## Data and a train/test split

As in [`05_knn`](05_knn.ipynb): fit on train, evaluate on test. `x1`, `x2` drive the label; `noise1`-`noise4` are pure noise, giving overfitting somewhere to hide.

In [1]:
import numpy as np
import polars as pl

from dolcestat.preprocessing import DolceSet
from dolcestat.trees import DecisionTree, RandomForest

rng = np.random.default_rng(0)
n = 400
x1 = rng.normal(0, 1, n)
x2 = rng.normal(0, 1, n)
noise = rng.normal(0, 1, (n, 4))
prob = 1 / (1 + np.exp(-(1.5 * x1 - 2.0 * x2 + 0.4)))
label = (rng.uniform(size=n) < prob).astype(int)

cols = {"x1": x1, "x2": x2}
cols.update({f"noise{i + 1}": noise[:, i] for i in range(4)})
cols["label"] = label
df = pl.DataFrame(cols)

order = rng.permutation(n)
train_rows, test_rows = order[:300].tolist(), order[300:].tolist()
train = DolceSet()
train.load_from_polars_dataframe(df[train_rows], target_col="label")
test = DolceSet()
test.load_from_polars_dataframe(df[test_rows], target_col="label")

## Fitting a single tree

Build a `DecisionTree`, `fit()`, then `predict` on the test set. `predict` returns an **analyzer** ([`06_metrics`](06_metrics.ipynb)): `y_fit` holds predictions, `accuracy`/`precision`/etc. read off it directly.

In [2]:
tree = DecisionTree(max_depth=4, min_samples_to_split=5).fit(train)
result = tree.predict(test)

print("predicted labels:", result.y_fit[:10])
print(f"test accuracy: {result.accuracy(0.5):.3f}")

predicted labels: [1 0 1 1 0 1 1 1 0 0]
test accuracy: 0.800


## The knobs

- **`max_depth`** — max split depth. Too shallow underfits; unlimited depth memorises the training set, noise included.
- **`min_samples_to_split`** — nodes below this size become leaves instead of splitting further.
- **`impurity_measure`** — `"gini"` (default) or `"entropy"` for classification, `"mse"` for regression.
- **`categorical_features`** — column indices split by *equality* (`feature == category`) instead of a numeric threshold.
- **class labels** — classification targets must be the integers `0, 1, …, K-1` (see `DolceSet.encode_target` in [`01_preprocessing`](01_preprocessing.ipynb)). With more than two classes `predict` returns a multiclass analyzer — see [`06_metrics`](06_metrics.ipynb).


In [3]:
for depth in [1, 2, 4, 8, None]:
    t = DecisionTree(max_depth=depth, min_samples_to_split=5).fit(train)
    train_acc = t.predict(train).accuracy(0.5)
    test_acc = t.predict(test).accuracy(0.5)
    print(f"max_depth={depth!s:>4}  train_acc={train_acc:.3f}  test_acc={test_acc:.3f}")

max_depth=   1  train_acc=0.767  test_acc=0.680


max_depth=   2  train_acc=0.777  test_acc=0.720


max_depth=   4  train_acc=0.853  test_acc=0.800


max_depth=   8  train_acc=0.963  test_acc=0.660


max_depth=None  train_acc=0.970  test_acc=0.660


### Categorical splits

`categorical_features` splits those columns by category membership instead of a threshold — useful when a numeric encoding (e.g. label-encoded `0`, `1`, `2`) carries no real order.

In [4]:
cat = rng.integers(0, 3, n).astype(float)  # 3 categories, label-encoded
cat_effect = np.where(cat == 0, 2.0, np.where(cat == 1, -1.0, 0.5))
cat_noise = rng.normal(0, 1, n)
cat_prob = 1 / (1 + np.exp(-(cat_effect + 0.3 * cat_noise)))
cat_label = (rng.uniform(size=n) < cat_prob).astype(int)
cat_df = pl.DataFrame({"category": cat, "noise": cat_noise, "label": cat_label})
cat_data = DolceSet()
cat_data.load_from_polars_dataframe(cat_df, target_col="label")

cat_tree = DecisionTree(max_depth=3, categorical_features=[0]).fit(cat_data)
print("train accuracy:", round(cat_tree.predict(cat_data).accuracy(0.5), 3))
print("root split is categorical:", cat_tree.tree.is_categorical_split)

train accuracy: 0.772
root split is categorical: True


## Random forests: bagging + feature subsampling

`RandomForest` grows `n_trees` trees, each on a **bootstrap sample** of rows and a **random subset of `n_feat` columns**. Different features per tree decorrelates their errors, so majority voting cancels more noise than any single tree suggests. `predict` returns the same analyzer as a single tree.

One simplification: textbook forests re-draw the column subset at **every split**; here it is drawn **once per tree**. That is simpler, and still decorrelates the trees. Forests draw from NumPy's global random generator, so the cells below seed it to be reproducible.


In [5]:
deep_tree = DecisionTree(max_depth=None, min_samples_to_split=2).fit(train)
print("single deep tree   train_acc=%.3f  test_acc=%.3f" % (
    deep_tree.predict(train).accuracy(0.5), deep_tree.predict(test).accuracy(0.5)))

np.random.seed(0)  # forests draw from NumPy's global random generator
rf = RandomForest(n_trees=100, n_feat=3, max_depth=None, min_samples_to_split=2).fit(train)
print("random forest       train_acc=%.3f  test_acc=%.3f" % (
    rf.predict(train).accuracy(0.5), rf.predict(test).accuracy(0.5)))

single deep tree   train_acc=1.000  test_acc=0.720


random forest       train_acc=1.000  test_acc=0.790


### More trees, less variance

Each tree stays free to overfit (`max_depth=None`). `n_trees` controls the *forest's* generalisation: more trees average away per-tree noise. The trend is up, but noisy on a 100-row test set.

In [6]:
for n_trees in [1, 5, 20, 100]:
    np.random.seed(0)
    forest = RandomForest(
        n_trees=n_trees, n_feat=3, max_depth=None, min_samples_to_split=2
    ).fit(train)
    test_acc = forest.predict(test).accuracy(0.5)
    print(f"n_trees={n_trees:>4}  test_acc={test_acc:.3f}")

n_trees=   1  test_acc=0.670


n_trees=   5  test_acc=0.690


n_trees=  20  test_acc=0.670


n_trees= 100  test_acc=0.790


## Regression trees

`impurity_measure="mse"` turns the same `DecisionTree`/`RandomForest` into regressors: splits are scored by variance reduction, leaves predict the mean target. `predict` returns a `RegressionAnalyzer` ([`06_metrics`](06_metrics.ipynb)) — read `rmse()`, `mape()`, etc.

New data: apartment price from size and room count (as in [`06_metrics`](06_metrics.ipynb)), plus three noise features.

In [7]:
reg_rng = np.random.default_rng(2)
m = 400
size_m2 = reg_rng.uniform(45, 190, m)
rooms = reg_rng.integers(1, 6, m).astype(float)
reg_noise = reg_rng.normal(0, 1, (m, 3))
price_k = 3.0 * size_m2 + 25 * rooms + 60 + reg_rng.normal(0, 18, m)

reg_cols = {"size_m2": size_m2, "rooms": rooms}
reg_cols.update({f"noise{i + 1}": reg_noise[:, i] for i in range(3)})
reg_cols["price_k"] = price_k
reg_df = pl.DataFrame(reg_cols)

reg_order = reg_rng.permutation(m)
reg_train = DolceSet()
reg_train.load_from_polars_dataframe(reg_df[reg_order[:300].tolist()], target_col="price_k")
reg_test = DolceSet()
reg_test.load_from_polars_dataframe(reg_df[reg_order[300:].tolist()], target_col="price_k")

reg_tree = DecisionTree(max_depth=4, min_samples_to_split=5, impurity_measure="mse").fit(reg_train)
reg_result = reg_tree.predict(reg_test)

print("predicted prices:", np.round(reg_result.y_fit[:6], 1))
print("true prices:     ", np.round(reg_result.y_true[:6], 1))
print(f"test RMSE: {reg_result.rmse():.2f} (thousand €)")
print(f"test MAPE: {reg_result.mape():.2f}%")

predicted prices: [596.4 501.3 668.9 369.7 385.4 515.7]
true prices:      [558.3 494.3 646.  399.4 429.4 505.5]
test RMSE: 32.42 (thousand €)
test MAPE: 5.41%


### Random forest regression — `n_feat` matters more here

Price here depends almost entirely on two features (`size_m2`, `rooms`) out of five. Too few columns per tree and some trees never see either informative feature, hurting the ensemble. Feature subsampling only pays off once `n_feat` gives most trees a real shot at the signal.

In [8]:
deep_reg_tree = DecisionTree(
    max_depth=None, min_samples_to_split=2, impurity_measure="mse"
).fit(reg_train)
print("single deep tree   test_rmse=%.2f" % deep_reg_tree.predict(reg_test).rmse())

for n_feat in [2, 3, 4, 5]:
    np.random.seed(0)
    reg_forest = RandomForest(
        n_trees=100,
        n_feat=n_feat,
        max_depth=None,
        min_samples_to_split=2,
        impurity_measure="mse",
    ).fit(reg_train)
    test_rmse = reg_forest.predict(reg_test).rmse()
    print(f"n_feat={n_feat}  random forest  test_rmse={test_rmse:.2f}")

single deep tree   test_rmse=29.83


n_feat=2  random forest  test_rmse=76.59


n_feat=3  random forest  test_rmse=55.79


n_feat=4  random forest  test_rmse=32.42


n_feat=5  random forest  test_rmse=19.15
